<a href="https://colab.research.google.com/github/DeKUT-DSAIL/MasharikiWeather/blob/main/docs/source/examples/masharikiweather_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MasharikiWeather Demo
This is an internal (private) Weather team (at DSAIL) notebook to demonstrate the progress, potential and intent of MasharikiWeather dataset.

This is the progress of the first version, it will be iteratively improved with time.

In [1]:
!pip install filter-stations==0.7.1 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 26.6 MB/s eta 0:00:00


## Setting up
One needs a specific Github token (provided upon request) to be able to clone the repo (in development) while running on colab.

One also requires a huggingface token to be able to access the data.

These are measures that have been put up because we do not have the rights to repackage some of the data products.

**N/B** The cloning might break at times but retrying it a couple of times would help (ensure to disconnect and delete runtime before each retry).

In [2]:
import os
from google.colab import userdata

# Securely fetch the GitHub token from Colab Secrets
github_token = userdata.get('COLAB_MASHARIKI')

# Define your organization and repository details
org_name = "DeKUT-DSAIL"
repo_name = "MasharikiWeather"

In [3]:
# Clone private repo
clone_url = f"https://{github_token}@github.com/{org_name}/{repo_name}.git"

print(f"Cloning private repository {org_name}/{repo_name}...")
exit_code = os.system(f"git clone {clone_url} -q")

if exit_code == 0:
    print("Clone successful!")
    # Change the working directory to the cloned repository
    os.chdir(repo_name)

    # Install the requirements
    print("Installing dependencies...")
    os.system("pip install -r requirements.txt -q")
    print("Environment ready.")
else:
    print("Error: Failed to clone repository. Check your GITHUB_TOKEN and organization SSO authorization.")

Cloning private repository DeKUT-DSAIL/MasharikiWeather...
Clone successful!
Installing dependencies...
Environment ready.


In [4]:
# !git clone https://github.com/DeKUT-DSAIL/MasharikiWeather.git

In [5]:
os.listdir()

['masharikiweather', '.git', 'README.md', 'requirements.txt', '.gitignore']

In [6]:
# Install dependencies
!pip install -r requirements.txt --quiet

In [7]:
mashariki_hf_token = userdata.get('MASHARIKI_HF_TOKEN') # Huggingface dataset token

In [8]:
# Log in on HF
from huggingface_hub import notebook_login
notebook_login()

In [9]:
import xarray as xr
import pandas as pd
import numpy as np
import zarr
import fsspec

from filter_stations import RetrieveData

In [10]:
# Use TAHMO credentials with filter stations
tahmo_api_key = userdata.get("TAHMO_API_KEY")
tahmo_api_secret = userdata.get("TAHMO_API_SECRET")

In [11]:
# Init retrieve class
retrieve_client = RetrieveData(tahmo_api_key, tahmo_api_secret)

In [12]:
station_info_ke = retrieve_client.get_stations_info(countrycode='KE')
station_info_ke.head()

,code,status,installationdate,elevationground,sensorinstallations,dataloggerinstallations,creatorid,created,updaterid,updated,...,location.countrycode,location.zipcode,location.latitude,location.longitude,location.elevationmsl,location.note,location.creatorid,location.created,location.updaterid,location.updated
0,TA00001,1,2012-08-01T00:00:00Z,2.0,None,None,2,2018-11-05T08:23:12.311416Z,199,2025-11-24T15:09:52.516117Z,...,KE,,-1.123283,34.397992,1393.0,Geen note,1,2018-10-26T05:11:52.099709Z,2,2018-10-26T12:15:59.662751Z
16,TA00018,1,2014-06-18T00:00:00Z,2.0,None,None,2,2018-11-07T16:16:48.249586Z,2,2018-11-07T16:16:48.249586Z,...,KE,,-0.582011,34.607261,1411.9,{},2,2018-10-26T13:12:06.136054Z,2,2022-07-26T07:23:14.526166Z
17,TA00019,1,2015-03-27T00:00:00Z,2.0,None,None,2,2018-11-09T14:41:40.385279Z,2,2018-11-09T14:49:51.109343Z,...,KE,,-0.524114,34.255647,1281.0,{},2,2018-10-26T13:13:40.215142Z,2,2018-10-26T13:13:40.215142Z
18,TA00020,1,2015-05-28T00:00:00Z,2.0,None,None,2,2018-11-09T14:57:40.121861Z,2,2018-11-09T14:57:40.121861Z,...,KE,,-1.653356,36.862397,1643.0,{},2,2018-10-26T13:16:36.222199Z,2,2018-10-26T13:16:36.222199Z
19,TA00021,1,2015-06-09T00:00:00Z,2.0,None,None,2,2018-11-09T15:25:56.547597Z,2,2018-11-09T15:25:56.547597Z,...,KE,,0.757986,35.173711,1948.4,{},2,2018-10-26T13:18:18.166819Z,2,2022-07-26T07:23:58.857542Z


In [13]:
# Using Kenyan station list for TAHMO
station_list = station_info_ke['code'].to_list()
print(station_list[:10])

['TA00001', 'TA00018', 'TA00019', 'TA00020', 'TA00021', 'TA00022', 'TA00023', 'TA00024', 'TA00025', 'TA00026']


In [14]:
# GhCNd stations
ghcnd_station_list_ea = [
    'KE000063612', 'KE000063619', 'KE000063624',
    'KE000063661', 'KE000063723', 'KE000063740',
    'KE000063820', 'KEM00063686','KEM00063741',
    'KEM00063799', 'MZM00067205', 'RWM00064387',
    'SU000062941', 'TZ000063756', 'TZ000063832',
    'TZ000063862','TZ000063894', 'TZ000063962',
    'TZM00063870','ZA000067475']
    # 'TZ000063971' - Ignore Mtwara Airport station


In [15]:
# TAHMO + GhCND stations
station_list.extend(ghcnd_station_list_ea)

In [16]:
from masharikiweather.dataset import MasharikiWeatherDataset

# Data Understanding
- Observations with their metadata (latitude, longitude, static priors)
- `Nearest Neighbor` and `Bilinear` interpolation methods of gridded products and station data.
- Demonstration of window generation for downstream ML tasks.

**N/B** Only precipitation is currently available for the station data but other variables will soon be included.

In [17]:
# Init the Dataset class
ds = MasharikiWeatherDataset(
    repo_id="DeKUT-DSAIL/weather-data",
    token=mashariki_hf_token,
    parameters=['total_precipitation'],
    source_obs=["tahmo", "ghcnd"],
    freq="h",
    years=[2020, 2021, 2022, 2023, 2024]
)

print("--- Station Metadata ---")
target_stations = station_list

# Get the station metadata
metadata_df = ds.get_stations_metadata(target_stations)
display(metadata_df)

--- Station Metadata ---


,latitude,longitude,network,dem,Slope,Aspect,LandCover,DistanceToWater,STD_2000M,TPI_2000M,WE_DERIVATIVE_2000M_SIGRATIO1,SN_DERIVATIVE_2000M_SIGRATIO1,ASPECT_2000M_SIGRATIO1,SLOPE_2000M_SIGRATIO1,STD_10000M,TPI_10000M,WE_DERIVATIVE_10000M_SIGRATIO1,SN_DERIVATIVE_10000M_SIGRATIO1,ASPECT_10000M_SIGRATIO1,SLOPE_10000M_SIGRATIO1
station_id,,,,,,,,,,,,,,,,,,,,
TA00001,-1.123283,34.397991,tahmo,1384.0,0.726879,175.425201,40.0,4000.000000,11.039467,-3.230769,-0.003853,-0.004591,221.044952,0.643444,47.401138,-17.523659,-0.003339,0.008998,185.777161,1.133147
TA00018,-0.582011,34.607262,tahmo,1399.0,1.503822,295.082458,20.0,14866.068359,31.054447,5.076923,-0.027651,-0.002409,268.342377,1.841831,81.115807,32.451103,-0.007762,0.010135,211.820419,1.436273
TA00019,-0.524114,34.255646,tahmo,1274.0,4.257180,121.096237,40.0,6708.204102,77.261253,-11.076923,0.053436,-0.028208,105.617065,3.811487,128.096100,8.968454,0.004728,0.010970,167.722488,3.127872
TA00020,-1.653356,36.862396,tahmo,1643.0,0.318955,90.000000,30.0,19235.384766,6.591190,0.307692,0.005216,-0.002451,110.851410,0.359474,26.082788,-0.977918,0.004985,0.001088,105.310287,0.481926
TA00021,0.757986,35.173710,tahmo,1955.0,0.522550,266.820435,40.0,5830.951660,18.602983,8.923077,-0.016220,0.007154,272.082703,1.221304,60.658897,20.372240,-0.009641,0.002154,260.474609,0.899746
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TZ000063862,-6.167000,35.767000,ghcnd,1114.0,0.458536,161.465149,50.0,4123.105469,9.000986,-3.538461,0.002935,0.007393,112.549133,0.632670,63.080505,-36.810726,0.000143,0.007726,164.389664,1.062021
TZ000063894,-6.867000,39.200000,ghcnd,59.0,0.287964,45.206219,50.0,9000.000000,4.375834,0.923077,0.005095,-0.000156,85.289925,0.350294,35.182659,-8.460567,0.006204,-0.001191,125.282150,0.566227
TZ000063962,-10.667000,35.583000,ghcnd,1034.0,0.992510,303.696808,40.0,44181.445312,15.030148,-11.307693,-0.009977,-0.000856,262.047638,0.890165,57.914459,-37.022083,-0.008155,0.002425,215.145004,1.358988


In [18]:
# Retrieve Ground Truth
obs_df, obs_mask = ds.get_observations(
    stations=station_list,
    parameters=['total_precipitation'],
    first_date='2020-01-01',
    last_date='2024-12-31',
    return_mask=True
)

display(obs_df.head())

station_id,TA00001,TA00018,TA00019,TA00020,TA00021,TA00023,TA00024,TA00025,TA00026,TA00028,...,MZM00067205,RWM00064387,SU000062941,TZ000063756,TZ000063832,TZ000063862,TZ000063894,TZ000063962,TZM00063870,ZA000067475
parameter,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,...,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation
time,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,0.000,0.0,0.0,0.000,0.0,0.0,0.017,0.034,0.000,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2020-01-01 01:00:00,0.034,0.0,0.0,0.017,0.0,0.0,0.017,0.034,0.000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 02:00:00,0.017,0.0,0.0,0.034,0.0,0.0,0.017,0.034,0.000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 03:00:00,0.034,0.0,0.0,0.034,0.0,0.0,0.017,0.034,0.000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-01 04:00:00,0.017,0.0,0.0,0.017,0.0,0.0,0.017,0.017,0.017,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


For the GhCNd stations, the valid value is at timestamp `00:00` of each available day. The other values are filled values (with zero), one can change the filling method. There is a binary mask to show the missingness, this mask would help in any model training, it would effectively tell the model to ignore the padded zeros, calculating its loss over the valid observations only.

In [19]:
# Retrieve Satellite/Reanalysis Context
print("\n--- Extracting Gridded Context ---")
gridded_context = ds.get_gridded_for_stations(
    groups=["era5", "imerg", "chirps", "tamsat"],
    stations=station_list,
    years=[2024],
    variables=['total_precipitation'],
    method="linear" # Interpolation method for the station - gridded product link
)


--- Extracting Gridded Context ---


Extracting tamsat: 100%|██████████| 1/1 [00:17<00:00, 17.57s/it]


In [20]:
# preview era5
gridded_context['era5'].head()

station_id,KE000063612,KE000063619,KE000063624,KE000063661,KE000063723,KE000063740,KE000063820,KEM00063686,KEM00063741,KEM00063799,...,TD00002,TD00004,TD00005,TZ000063756,TZ000063832,TZ000063862,TZ000063894,TZ000063962,TZM00063870,ZA000067475
parameter,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,...,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation
time,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,0.001818,0.0,0.0,0.024628,0.0,0.025444,0.000755,0.001685,0.000000,0.000033,...,0.001641,0.003799,0.001660,0.000000,0.013618,0.0,0.000000,0.000212,0.000393,0.003229
2024-01-01 01:00:00,0.013338,0.0,0.0,0.179668,0.0,0.024769,0.009438,0.004225,0.000095,0.000000,...,0.005473,0.033928,0.001580,0.000042,0.018151,0.0,0.000203,0.000000,0.009327,0.005823
2024-01-01 02:00:00,0.025946,0.0,0.0,0.406723,0.0,0.014037,0.031155,0.010848,0.007915,0.000000,...,0.018054,0.028864,0.008177,0.000475,0.052896,0.0,0.005911,0.000000,0.045237,0.016948
2024-01-01 03:00:00,0.007172,0.0,0.0,0.488121,0.0,0.007528,0.011678,0.004385,0.006962,0.000178,...,0.017209,0.009336,0.006978,0.001862,0.043471,0.0,0.043312,0.000792,0.258565,0.034354
2024-01-01 04:00:00,0.000000,0.0,0.0,0.527420,0.0,0.003197,0.010713,0.025770,0.002480,0.000016,...,0.039570,0.010635,0.002523,0.001036,0.000000,0.0,0.085302,0.000317,0.262755,0.035323


In [22]:
# preview Imerg
gridded_context['imerg'].head()

station_id,KE000063612,KE000063619,KE000063624,KE000063661,KE000063723,KE000063740,KE000063820,KEM00063686,KEM00063741,KEM00063799,...,TD00002,TD00004,TD00005,TZ000063756,TZ000063832,TZ000063862,TZ000063894,TZ000063962,TZM00063870,ZA000067475
parameter,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,...,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation
time,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.002739
2024-01-01 00:30:00,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0000,0.0,...,0.090283,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
2024-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.011979,0.0,0.092115,0.0600,0.0,...,0.552403,0.0,0.057302,0.0,0.0,0.0,0.0,0.0,0.0,0.001708
2024-01-01 01:30:00,0.0,0.0,0.0,0.0,0.0,0.013613,0.0,0.275945,0.0625,0.0,...,1.263237,0.0,0.060201,0.0,0.0,0.0,0.0,0.0,0.0,0.061648
2024-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.015791,0.0,0.339070,0.0700,0.0,...,1.115038,0.0,0.067671,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


In [23]:
# preview chirps
gridded_context['chirps'].head()

station_id,KE000063612,KE000063619,KE000063624,KE000063661,KE000063723,KE000063740,KE000063820,KEM00063686,KEM00063741,KEM00063799,...,TD00002,TD00004,TD00005,TZ000063756,TZ000063832,TZ000063862,TZ000063894,TZ000063962,TZM00063870,ZA000067475
parameter,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,...,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation,total_precipitation
time,,,,,,,,,,,,,,,,,,,,,
2024-01-01,4.34747,0.0,0.0,0.000000,0.0,0.000000,0.0,35.010767,0.000000,0.0,...,47.586822,0.000000,0.000000,0.000000,2.693277,16.440197,0.000000,25.965792,NaN,6.816984
2024-01-02,0.00000,0.0,0.0,4.931826,0.0,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,4.515327,3.879312,27.400329,3.312649,33.778982,NaN,9.105746
2024-01-03,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,15.701549,5.386553,8.295875,0.000000,25.965792,NaN,3.408492
2024-01-04,0.00000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.142280,3.259171,0.000000,32.365503,27.597090,NaN,3.408492
2024-01-05,0.00000,0.0,0.0,0.000000,0.0,10.164259,0.0,0.000000,64.639656,0.0,...,0.000000,4.240845,62.847164,4.515327,0.000000,10.960131,28.800300,22.955520,NaN,6.816984


In [24]:
# Get the satellite/renalsyis products (lazy loading)
raw_datasets = ds.load_gridded_datasets(groups=["era5"],
                                        years=[2020, 2021, 2022, 2023, 2024])
print(raw_datasets.keys())
print(raw_datasets['era5'])

dict_keys(['era5'])
<xarray.Dataset> Size: 59GB
Dimensions:                       (time: 43848, lat: 69, lon: 57, level: 13)
Coordinates:
  * time                          (time) datetime64[ns] 351kB 2020-01-01 ... ...
  * lat                           (lat) float64 552B 5.0 4.75 ... -11.75 -12.0
  * lon                           (lon) float64 456B 28.0 28.25 ... 41.75 42.0
  * level                         (level) int64 104B 1000 925 850 ... 150 100 50
    number                        int64 8B ...
    step                          timedelta64[ns] 8B ...
Data variables: (12/14)
    10m_u_component_of_wind       (time, lat, lon) float32 690MB dask.array<chunksize=(48, 69, 57), meta=np.ndarray>
    10m_v_component_of_wind       (time, lat, lon) float32 690MB dask.array<chunksize=(48, 69, 57), meta=np.ndarray>
    2m_temperature                (time, lat, lon) float32 690MB dask.array<chunksize=(48, 69, 57), meta=np.ndarray>
    geopotential                  (time, level, lat, lon) float

In [25]:
# load to physical RAM
raw_datasets['era5'][['total_precipitation']].compute()

<xarray.Dataset> Size: 690MB
Dimensions:              (time: 43848, lat: 69, lon: 57)
Coordinates:
  * time                 (time) datetime64[ns] 351kB 2020-01-01 ... 2024-12-3...
  * lat                  (lat) float64 552B 5.0 4.75 4.5 ... -11.5 -11.75 -12.0
  * lon                  (lon) float64 456B 28.0 28.25 28.5 ... 41.5 41.75 42.0
    number               int64 8B 0
    step                 timedelta64[ns] 8B 00:00:00
Data variables:
    total_precipitation  (time, lat, lon) float32 690MB 0.0 0.0 ... 0.1469

In [26]:
# Showcase windows logic
sample_stations = ds.stations.to_list()[:20]
window_vars = ['total_precipitation'] # Adjust based on available TAHMO parameters
start_date = "2020-01-01"
end_date = "2024-12-31"

window_lookback = 24  # 24 hours of historical context
forecast_horizon = 6  # 6 hours of prediction

In [27]:
# Standard Observation Tensors (NumPy)
print("\n--- Standard Numpy Tensors ---")
np_windows = ds.get_windows(
    window_size=window_lookback,
    horizon_size=forecast_horizon,
    stations=sample_stations,
    parameters=window_vars,
    first_date=start_date,
    last_date=end_date,
    as_xarray=False
)

print(f"Features (X) shape:      {np_windows.x.shape}")
print(f"Targets (Y) shape:       {np_windows.y.shape}")
print(f"Feature Mask shape:      {np_windows.mask_x.shape}")
print(f"Target Mask shape:       {np_windows.mask_y.shape}")
print(f"Index X (Reftime) shape: {np_windows.index_x.shape}")


--- Standard Numpy Tensors ---
Features (X) shape:      (43795, 24, 20, 1)
Targets (Y) shape:       (43795, 6, 20, 1)
Feature Mask shape:      (43795, 24, 20, 1)
Target Mask shape:       (43795, 6, 20, 1)
Index X (Reftime) shape: (43795, 24)


In [28]:
# Xarray Tensors (Ready for Gridded Alignment)
print("\n--- Xarray Tensors ---")
xr_windows = ds.get_windows(
    window_size=window_lookback,
    horizon_size=forecast_horizon,
    stations=sample_stations,
    parameters=window_vars,
    first_date=start_date,
    last_date=end_date,
    as_xarray=True
)

print("Xarray Dataset (X) Coordinates:")
print(list(xr_windows.x.coords.keys()))
print(f"Xarray Dataset (X) Dimensions: {dict(xr_windows.x.dims)}")
print(f"Xarray Dataset (Y) Dimensions: {dict(xr_windows.y.dims)}")


--- Xarray Tensors ---
Xarray Dataset (X) Coordinates:
['reftime', 'lag', 'station']
Xarray Dataset (X) Dimensions: {'reftime': 43795, 'lag': 24, 'station': 20}
Xarray Dataset (Y) Dimensions: {'reftime': 43795, 'lead': 6, 'station': 20}


In [29]:
xr_windows.x

<xarray.Dataset> Size: 169MB
Dimensions:              (reftime: 43795, lag: 24, station: 20)
Coordinates:
  * reftime              (reftime) datetime64[ns] 350kB 2020-01-01T23:00:00 ....
  * lag                  (lag) timedelta64[ns] 192B -1 days +01:00:00 ... 0 d...
  * station              (station) object 160B 'TA00001' 'TA00019' ... 'TA00074'
Data variables:
    total_precipitation  (reftime, lag, station) float64 168MB 0.0 0.0 ... 0.0